# Elora tomato leaf disease: EfficientNetB0 experiment

**Issue 45 follow-up | Dev 1 | Kaggle experiment**

This notebook repeats the exact dataset audit and split from the [MobileNetV2 baseline](tomato_disease_baseline.ipynb), then tests EfficientNetB0 with training-only augmentation and a short fine-tuning stage. The baseline's recorded validation accuracy was 73.3%; its held-out test accuracy was 66.4% and macro F1 was 0.626. Do not use those test numbers to select this model. This is still an interim experiment, with no agronomy label review, independent Sri Lankan field set, calibrated confidence, or unsupported-input rejection.

Run from top to bottom in Kaggle with Internet and a GPU enabled. The archive is downloaded and audited again so this notebook is self-contained. Save its version and download its JSON, training CSV files, and selected `.keras` model afterward. Do not commit the model weights to ordinary Git.


## 1. Acquire and verify the source archive

The source reports 7,200 images. The archive contains 7,200 **augmented** images plus 830 files in `Dataset (raw)`. This run trains and evaluates only on audited raw files, because augmented copies can leak across splits. The SHA-256 below identifies the exact archive used here. If the source changes, stop and repeat the audit before accepting different counts or a different hash.


In [ ]:
from pathlib import Path
import hashlib
import requests
import zipfile

url = (
    "https://data.mendeley.com/public-files/datasets/3mbnb82mxd/"
    "files/4844fda3-b66b-4b2a-bd83-775c0da974ea/file_downloaded"
)
archive = Path("/kaggle/working/Tomato_Leaf_Disease_Dataset_Pakistan_v2.zip")

response = requests.get(url, stream=True, timeout=60)
response.raise_for_status()

sha256 = hashlib.sha256()
with archive.open("wb") as output:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
        if chunk:
            output.write(chunk)
            sha256.update(chunk)

print("Downloaded bytes:", archive.stat().st_size)
print("SHA-256:", sha256.hexdigest())
print("Valid ZIP:", zipfile.is_zipfile(archive))
assert archive.stat().st_size == 726457628
assert sha256.hexdigest() == "2a2b36a8f972337f6d0e95e436835e55ffee5291fa07277157b40f0441f42f50"
assert zipfile.is_zipfile(archive)

## 2. Inspect archive structure, documentation, and image labels

Inspect folder counts before selecting a training set. The README and licence declare six leaf classes, field collection in Pakistan, manual visual labels, and CC BY 4.0. The PDF describes 7,200 images but does not clearly separate raw files from augmented copies. A licence statement is recorded as source provenance; image-by-image rights have not been verified.


In [ ]:
from collections import Counter
from pathlib import PurePosixPath
import zipfile

with zipfile.ZipFile(archive) as z:
    files = [name for name in z.namelist() if not name.endswith("/")]

print("Files:", len(files))
print("File types:", Counter(PurePosixPath(name).suffix.lower() for name in files))
print("\nFirst 20 paths:")
for name in files[:20]:
    print(name)

In [ ]:
from collections import Counter
from pathlib import PurePosixPath

image_suffixes = {".jpg", ".jpeg"}
image_paths = [PurePosixPath(name) for name in files
               if PurePosixPath(name).suffix.lower() in image_suffixes]

folders = Counter("/".join(path.parts[:-1]) for path in image_paths)

for folder, count in sorted(folders.items()):
    print(f"{count:5}  {folder}")

print("\nOther files:")
for name in files:
    if PurePosixPath(name).suffix.lower() not in image_suffixes:
        print(name)

In [ ]:
with zipfile.ZipFile(archive) as z:
    for name in [
        "Tomato Dataset/README.md",
        "Tomato Dataset/License.txt",
    ]:
        print(f"\n--- {name} ---")
        print(z.read(name).decode("utf-8", errors="replace")[:6000])

In [ ]:
%pip -q install pypdf ImageHash

In [ ]:
from io import BytesIO
from pypdf import PdfReader

with zipfile.ZipFile(archive) as z:
    pdf_bytes = z.read("Tomato Dataset/Dataset_Description.pdf")

reader = PdfReader(BytesIO(pdf_bytes))
print("Pages:", len(reader.pages))

for page_number, page in enumerate(reader.pages, start=1):
    print(f"\n--- Page {page_number} ---")
    print((page.extract_text() or "")[:8000])

### Visual source check

Sample each source folder to spot obvious wrong-organ, unusable, or identifier-bearing photos. Disease-label judgments require an agronomy reviewer, who is unavailable for this interim run. Keep that limitation in the evaluation record.


In [ ]:
import random
from collections import defaultdict
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

rng = random.Random(45)
raw_by_class = defaultdict(list)

for name in files:
    path = PurePosixPath(name)
    if "Dataset (raw)" in path.parts and path.suffix.lower() in {".jpg", ".jpeg"}:
        raw_by_class[path.parent.name].append(name)

fig, axes = plt.subplots(len(raw_by_class), 4, figsize=(12, 16))

with zipfile.ZipFile(archive) as z:
    for row, (label, paths) in enumerate(sorted(raw_by_class.items())):
        for col, name in enumerate(rng.sample(paths, 4)):
            image = Image.open(BytesIO(z.read(name))).convert("RGB")
            axes[row, col].imshow(image)
            axes[row, col].set_title(f"{label}\n{PurePosixPath(name).name[:28]}", fontsize=8)
            axes[row, col].axis("off")

plt.tight_layout()
plt.show()

## 3. Audit the raw images

First find **exact duplicates** using SHA-256 of image bytes. If an identical image appears under conflicting labels, exclude every copy. For duplicates within one label, retain one deterministic path. Then check that retained images decode. These choices are recorded in the manifest.


In [ ]:
from collections import defaultdict
import hashlib

paths_by_hash = defaultdict(list)

with zipfile.ZipFile(archive) as z:
    for paths in raw_by_class.values():
        for name in paths:
            digest = hashlib.sha256(z.read(name)).hexdigest()
            paths_by_hash[digest].append(name)

duplicate_groups = [
    paths for paths in paths_by_hash.values() if len(paths) > 1
]

print("Raw files:", sum(len(paths) for paths in raw_by_class.values()))
print("Unique file hashes:", len(paths_by_hash))
print("Exact duplicate groups:", len(duplicate_groups))

for group in duplicate_groups[:10]:
    print("\n", *group, sep="\n  ")

In [ ]:
from pathlib import PurePosixPath

conflicting_groups = []
same_label_groups = []

for paths in duplicate_groups:
    labels = {PurePosixPath(path).parent.name for path in paths}
    if len(labels) > 1:
        conflicting_groups.append(paths)
    else:
        same_label_groups.append(paths)

print("Duplicate groups with conflicting labels:", len(conflicting_groups))
print("Duplicate groups within one label:", len(same_label_groups))

for group in conflicting_groups:
    print("\n" + "\n".join(group))

In [ ]:
from collections import Counter

usable_paths = []
excluded = {}

for paths in paths_by_hash.values():
    labels = {PurePosixPath(path).parent.name for path in paths}

    if len(labels) > 1:
        for path in paths:
            excluded[path] = "conflicting labels on identical image"
        continue

    keep = sorted(paths)[0]
    usable_paths.append(keep)

    for path in paths:
        if path != keep:
            excluded[path] = f"exact duplicate of {keep}"

counts = Counter(PurePosixPath(path).parent.name for path in usable_paths)

print("Candidate originals:", len(usable_paths))
print("Excluded:", len(excluded))
for label, count in sorted(counts.items()):
    print(label, count)

In [ ]:
from collections import Counter
from io import BytesIO
from PIL import Image

valid_paths = []
invalid_paths = []
sizes = Counter()

with zipfile.ZipFile(archive) as z:
    for path in usable_paths:
        try:
            with Image.open(BytesIO(z.read(path))) as image:
                image.load()
                sizes[image.size] += 1
            valid_paths.append(path)
        except Exception as error:
            invalid_paths.append((path, type(error).__name__))

print("Decoded successfully:", len(valid_paths))
print("Failed to decode:", len(invalid_paths))
print("Most common image sizes:", sizes.most_common(10))
print("Failures:", invalid_paths[:20])

### Review visually similar files

Perceptual hash proposes possible near duplicates; a small hash distance alone is not proof. Inspect every proposed pair side by side. The 11 pairs below were visually reviewed for this exact archive and all showed the same photo. The explicit reviewed-pair list makes the notebook stop if a future archive produces different pairs. For another crop, replace this list only after its own visual review.


In [ ]:
import imagehash
from itertools import combinations

visual_hashes = {}

with zipfile.ZipFile(archive) as z:
    for path in valid_paths:
        with Image.open(BytesIO(z.read(path))) as image:
            visual_hashes[path] = imagehash.phash(image.convert("RGB"))

near_pairs = []

for first, second in combinations(valid_paths, 2):
    distance = visual_hashes[first] - visual_hashes[second]
    if distance <= 4:
        near_pairs.append((distance, first, second))

near_pairs.sort()
print("Pairs needing visual review:", len(near_pairs))

for distance, first, second in near_pairs[:20]:
    print(f"\nDistance {distance}")
    print(first)
    print(second)

In [ ]:
import matplotlib.pyplot as plt

assert near_pairs, "No candidate pairs to display; inspect the audit before continuing."
fig, axes = plt.subplots(len(near_pairs), 2, figsize=(12, 3 * len(near_pairs)))

with zipfile.ZipFile(archive) as z:
    for row, (distance, first, second) in enumerate(near_pairs):
        for col, path in enumerate((first, second)):
            with Image.open(BytesIO(z.read(path))) as image:
                axes[row, col].imshow(image.convert("RGB"))

            axes[row, col].set_title(
                f"Distance {distance} | {PurePosixPath(path).parent.name}\n"
                f"{PurePosixPath(path).name[:55]}",
                fontsize=9,
            )
            axes[row, col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# These pairs were checked visually in the version 2 tomato archive.
root = "Tomato Dataset/Dataset (raw)/"
reviewed_pairs = {
    frozenset((root + first, root + second))
    for first, second in [
        ("Tomato_Early_blight_leaf/EBTomato.jpg", "Tomato_Early_blight_leaf/tomatoblight.jpg"),
        ("Tomato_Early_blight_leaf/EB_(973).jpg", "Tomato_leaf_late_blight/tomato_late-blight_04_zoom.jpg"),
        ("Tomato_Early_blight_leaf/EB_(992).jpg", "Tomato_Early_blight_leaf/images.jpg"),
        ("Tomato_Healthy_leaf/HL_(36).jpg", "Tomato_Healthy_leaf/images (1).jpg"),
        ("Tomato_Septoria_leaf_spot/FS547-4-big.jpg", "Tomato_Septoria_leaf_spot/Sept9-300x225.jpg"),
        ("Tomato_Septoria_leaf_spot/file96924.jpg", "Tomato_Septoria_leaf_spot/Tomato-Early-Blight-Leaves-Cornell-University.jpg"),
        ("Tomato_leaf_late_blight/924977LateBlightLeaf.jpg", "Tomato_leaf_late_blight/LateBlightLeaf.jpg"),
        ("Tomato_Septoria_leaf_spot/5c77186a4ea326e5495c9d03cd89ce16.jpg", "Tomato_Septoria_leaf_spot/eb56b488e349c2de423ab6afbcc4bd5b_jpg.rf.55285dbd0f68011aa303f772d65a0a4d.jpg"),
        ("Tomato_Septoria_leaf_spot/Septoria_Leaf_Spot_of_Tomato2241-450x300.jpg", "Tomato_Septoria_leaf_spot/SS_ (974).jpg"),
        ("Tomato_leaf_yellow_curl_virus/ab40cce958c60d8b6e0ec2e4fdb8f526_jpg.rf.fc54d8302023bbae771a3d7a6aecbf89.jpg", "Tomato_leaf_yellow_curl_virus/Tomato2.jpg"),
        ("Tomato_leaf_yellow_curl_virus/tomato-plant-virus-tomato-yellow-leaf-curl-virus-tomato-plant-disease-mosaic-virus.jpg", "Tomato_leaf_yellow_curl_virus/TYLCVDFerrinlr.jpg"),
    ]
}
assert {frozenset((first, second)) for _, first, second in near_pairs} == reviewed_pairs

near_excluded = {}

# The 11 reviewed pairs do not overlap, so each path occurs in one pair.
pair_paths = [path for _, first, second in near_pairs
              for path in (first, second)]
assert len(pair_paths) == len(set(pair_paths))

for distance, first, second in near_pairs:
    first_label = PurePosixPath(first).parent.name
    second_label = PurePosixPath(second).parent.name

    if first_label != second_label:
        near_excluded[first] = "same photo with conflicting near-duplicate label"
        near_excluded[second] = "same photo with conflicting near-duplicate label"
    else:
        keep, remove = sorted((first, second))
        near_excluded[remove] = f"near-duplicate of {keep}"

final_paths = [path for path in valid_paths if path not in near_excluded]
all_exclusions = {**excluded, **near_excluded}

print("Near-duplicate exclusions:", len(near_excluded))
print("Total exclusions:", len(all_exclusions))
print("Candidate originals for split:", len(final_paths))

for label, count in sorted(
    Counter(PurePosixPath(path).parent.name for path in final_paths).items()
):
    print(label, count)

### Check whether images can be grouped by capture session

Images from the same plant or photo session should stay in one split. This archive has no usable EXIF capture dates or plant/session IDs for grouping. Some camera model names exist but do not identify individual plants. The later stratified split therefore reduces known exact/near-duplicate leakage but cannot rule out related-photo leakage.


In [ ]:
from collections import Counter

capture_dates = Counter()
camera_models = Counter()
images_with_dates = 0

with zipfile.ZipFile(archive) as z:
    for path in final_paths:
        with Image.open(BytesIO(z.read(path))) as image:
            exif = image.getexif()
            captured = exif.get(36867)  # DateTimeOriginal
            model = exif.get(272)        # Camera model

        if captured:
            images_with_dates += 1
            capture_dates[captured[:10]] += 1
        if model:
            camera_models[str(model)] += 1

print("Images with EXIF capture date:", images_with_dates, "of", len(final_paths))
print("Most common capture dates:", capture_dates.most_common(10))
print("Camera models:", camera_models.most_common(10))

## 4. Make one reproducible split and manifest

Use raw, decoded, deduplicated paths only. Stratify by source label with seed 45 to produce approximately 70% training, 15% validation, and 15% test. Training fits model parameters, validation chooses the checkpoint, and the test set is touched only after training. The 116 test images estimate performance on this held-out portion of the same dataset, not real-world field accuracy.

The manifest records every raw path, hash, source label, normalized label, split, and exclusion reason. `group_id` is blank because no reliable group identifier was available. The assertions below are specific to this archive and are audit checkpoints, not generic target sizes.


In [ ]:
from sklearn.model_selection import train_test_split

def labels_for(paths):
    return [PurePosixPath(path).parent.name for path in paths]

train_val_paths, test_paths = train_test_split(
    final_paths,
    test_size=0.15,
    stratify=labels_for(final_paths),
    random_state=45,
)

train_paths, val_paths = train_test_split(
    train_val_paths,
    test_size=0.15 / 0.85,
    stratify=labels_for(train_val_paths),
    random_state=45,
)

for split_name, paths in [
    ("train", train_paths),
    ("validation", val_paths),
    ("test", test_paths),
]:
    print(split_name, len(paths))
    print(dict(sorted(Counter(labels_for(paths)).items())))

In [ ]:
import csv

label_map = {
    "Tomato_Early_blight_leaf": "early_blight",
    "Tomato_Healthy_leaf": "healthy",
    "Tomato_Septoria_leaf_spot": "septoria_leaf_spot",
    "Tomato_leaf_late_blight": "late_blight",
    "Tomato_leaf_yellow_curl_virus": "yellow_leaf_curl_virus",
    "Tomato_mold_leaf": "leaf_mold",
}

split_by_path = {}
for split_name, paths in [
    ("train", train_paths),
    ("validation", val_paths),
    ("test", test_paths),
]:
    for path in paths:
        split_by_path[path] = split_name

hash_by_path = {
    path: digest
    for digest, paths in paths_by_hash.items()
    for path in paths
}

manifest_path = Path("/kaggle/working/dataset_manifest.csv")
columns = [
    "source_id", "original_path", "sha256", "crop", "organ",
    "source_label", "label_id", "image_origin", "group_id",
    "split", "exclusion_reason",
]

with manifest_path.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=columns)
    writer.writeheader()

    for paths in raw_by_class.values():
        for path in sorted(paths):
            source_label = PurePosixPath(path).parent.name
            writer.writerow({
                "source_id": "10.17632/3mbnb82mxd.2",
                "original_path": path,
                "sha256": hash_by_path[path],
                "crop": "tomato",
                "organ": "leaf",
                "source_label": source_label,
                "label_id": label_map[source_label],
                "image_origin": "raw",
                "group_id": "",
                "split": split_by_path.get(path, "excluded"),
                "exclusion_reason": all_exclusions.get(path, ""),
            })

print(manifest_path)
print("Rows:", sum(1 for _ in manifest_path.open(encoding="utf-8")) - 1)

In [ ]:
from collections import Counter

with manifest_path.open(newline="", encoding="utf-8") as source:
    rows = list(csv.DictReader(source))

split_counts = Counter(row["split"] for row in rows)
paths = [row["original_path"] for row in rows]

print("Split counts:", dict(split_counts))
print("Unique paths:", len(set(paths)))
print("Rows missing a hash:", sum(not row["sha256"] for row in rows))
print("Excluded rows missing a reason:", sum(
    row["split"] == "excluded" and not row["exclusion_reason"]
    for row in rows
))

assert split_counts == {
    "train": 538,
    "validation": 116,
    "test": 116,
    "excluded": 60,
}
assert len(paths) == len(set(paths)) == 830
assert all(row["sha256"] for row in rows)
assert all(
    row["exclusion_reason"]
    for row in rows if row["split"] == "excluded"
)

## 5. Prepare image batches

Kaggle can train on GPU if one is available. Extract only the 770 selected files to temporary storage. Decode each JPEG with three color channels, resize with padding to 224 × 224, batch 16 images, and prefetch. Pixels remain in the 0–255 range here; normalization is inside the model, so inference must use the same model input convention.


In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

In [ ]:
data_root = Path("/kaggle/temp/elora_issue45")
data_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive) as z:
    for path in final_paths:
        z.extract(path, data_root)

extracted_paths = [data_root / path for path in final_paths]

print("Expected images:", len(final_paths))
print("Extracted images:", sum(path.is_file() for path in extracted_paths))
print("Example:", extracted_paths[0])

In [ ]:
import tensorflow as tf

class_names = [
    "early_blight",
    "late_blight",
    "septoria_leaf_spot",
    "leaf_mold",
    "yellow_leaf_curl_virus",
    "healthy",
]
label_to_index = {name: index for index, name in enumerate(class_names)}

def make_dataset(split, shuffle=False):
    selected = [row for row in rows if row["split"] == split]
    paths = [str(data_root / row["original_path"]) for row in selected]
    labels = [label_to_index[row["label_id"]] for row in selected]

    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(paths), seed=45)

    def load_image(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        image = tf.image.resize_with_pad(image, 224, 224)
        return image, label

    return (
        dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
        .batch(16)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = make_dataset("train", shuffle=True)
val_ds = make_dataset("validation")

for images, labels in train_ds.take(1):
    print("Image batch:", images.shape)
    print("Label batch:", labels.shape)
    print("Pixel range:", float(tf.reduce_min(images)), float(tf.reduce_max(images)))

## 6. Build the EfficientNetB0 feature model

The six output classes and 224 × 224 input stay the same as the baseline. ImageNet weights provide starting image features. Training augmentation applies random horizontal flips, small rotations/zooms, and contrast changes **only during fitting**. Validation and test images are not augmented. Keras EfficientNetB0 includes its own input preprocessing and expects 0–255 pixels, so this model must not add MobileNetV2's −1–1 rescaling.


In [ ]:
from tensorflow import keras

keras.utils.set_random_seed(45)

augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.05),
    keras.layers.RandomZoom(0.10),
    keras.layers.RandomContrast(0.10),
], name="train_only_augmentation")

base_model = keras.applications.EfficientNetB0(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = augmentation(inputs)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(len(class_names), activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)
model.summary()


## 7. Train the new head

First keep EfficientNetB0 frozen. Save the checkpoint with lowest validation loss and stop early if validation loss stops improving. This stage tests whether its fixed features separate the six labels better than the MobileNetV2 head did.


In [ ]:
head_checkpoint = "/kaggle/working/tomato_efficientnet_head.keras"
head_history_path = "/kaggle/working/efficientnet_head_history.csv"

head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            head_checkpoint, monitor="val_loss", save_best_only=True
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
        keras.callbacks.CSVLogger(head_history_path),
    ],
)


## 8. Fine-tune a small part of the backbone

Reload the best head checkpoint. Unfreeze only the last 30 EfficientNet layers, while keeping batch-normalization layers frozen. Use a much smaller learning rate so the pretrained features change gradually. This stage may improve or worsen validation performance; the later validation comparison decides whether to keep it.


In [ ]:
model = keras.models.load_model(head_checkpoint)
base_model = model.get_layer("efficientnetb0")
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers[-30:]:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

fine_checkpoint = "/kaggle/working/tomato_efficientnet_finetuned.keras"
fine_history_path = "/kaggle/working/efficientnet_finetune_history.csv"

fine_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[
        keras.callbacks.ModelCheckpoint(
            fine_checkpoint, monitor="val_loss", save_best_only=True
        ),
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=3, restore_best_weights=True
        ),
        keras.callbacks.CSVLogger(fine_history_path),
    ],
)


## 9. Choose a checkpoint on validation data

Calculate macro F1 for both saved checkpoints on the 116 validation originals. Macro F1 gives each class equal weight, so weak yellow leaf curl performance cannot hide behind larger classes. Choose the higher macro F1; use accuracy only to break a tie. This is model selection, so **do not inspect the test set in this step**.


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

val_rows = [row for row in rows if row["split"] == "validation"]
val_true = np.array([label_to_index[row["label_id"]] for row in val_rows])

validation_results = {}
for stage, path in [("head", head_checkpoint), ("fine_tuned", fine_checkpoint)]:
    candidate = keras.models.load_model(path)
    predictions = candidate.predict(val_ds).argmax(axis=1)
    validation_results[stage] = {
        "checkpoint": path,
        "macro_f1": float(f1_score(val_true, predictions, average="macro")),
        "accuracy": float(accuracy_score(val_true, predictions)),
    }

chosen_stage = max(
    validation_results,
    key=lambda stage: (
        validation_results[stage]["macro_f1"],
        validation_results[stage]["accuracy"],
    ),
)
chosen_path = validation_results[chosen_stage]["checkpoint"]
print("Validation results:", validation_results)
print("Selected stage:", chosen_stage)


## 10. Evaluate the selected checkpoint once

The test set was already inspected in the earlier MobileNetV2 run. These new test numbers are useful as an **exploratory comparison**, but they are not a fresh unbiased final test. A later independently collected Sri Lankan phone-photo set is needed for that claim. Report every class and the confusion matrix, including weak classes.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

selected_model = keras.models.load_model(chosen_path)
test_ds = make_dataset("test")
test_rows = [row for row in rows if row["split"] == "test"]
test_true = np.array([label_to_index[row["label_id"]] for row in test_rows])
test_scores = selected_model.predict(test_ds)
test_predicted = test_scores.argmax(axis=1)

print(classification_report(
    test_true, test_predicted, target_names=class_names,
    digits=3, zero_division=0,
))
print("Confusion matrix:")
print(confusion_matrix(test_true, test_predicted))


## 11. Save evidence and artifact checksum

Write metrics and model selection to JSON, copy the chosen checkpoint to one clearly named file, and calculate its SHA-256. The checksum identifies exact weights. Raw softmax scores remain uncalibrated, and this notebook has no fruit/domain rejection. Accuracy alone cannot establish a confident diagnosis.


In [ ]:
import json
import shutil

selected_path = Path("/kaggle/working/tomato_efficientnet_selected.keras")
shutil.copyfile(chosen_path, selected_path)
selected_sha256 = hashlib.sha256(selected_path.read_bytes()).hexdigest()

evaluation = {
    "source_doi": "10.17632/3mbnb82mxd.2",
    "archive_sha256": sha256.hexdigest(),
    "model": "EfficientNetB0 ImageNet features; head training and partial fine-tuning",
    "class_order": class_names,
    "split_seed": 45,
    "split_counts": dict(split_counts),
    "validation_results": validation_results,
    "chosen_stage": chosen_stage,
    "selected_model_sha256": selected_sha256,
    "test_classification_report": classification_report(
        test_true, test_predicted, target_names=class_names,
        output_dict=True, zero_division=0,
    ),
    "test_confusion_matrix": confusion_matrix(test_true, test_predicted).tolist(),
    "limitations": [
        "Agronomy label review unavailable for this interim experiment.",
        "Plant/session grouping unavailable; related-photo leakage remains possible.",
        "Test set was inspected after the earlier baseline; this is an exploratory comparison.",
        "No independent Sri Lankan field evaluation or unsupported-input challenge set.",
        "Softmax scores are uncalibrated; the 60% disease cutoff was not assessed.",
    ],
}

evaluation_path = Path("/kaggle/working/efficientnet_evaluation.json")
evaluation_path.write_text(json.dumps(evaluation, indent=2), encoding="utf-8")
for path in [evaluation_path, Path(head_history_path), Path(fine_history_path), selected_path]:
    print(path.name, path.stat().st_size, "bytes")
print("Selected model SHA-256:", selected_sha256)


## What to compare with the first baseline

Record the new validation macro F1 and test per-class metrics alongside the earlier MobileNetV2 run. A higher validation score supports further study; it does not erase uncertain labels or prove field accuracy. If fine-tuning performs worse on validation, retain the head-only checkpoint. The selected model must still pass calibration, fruit/wrong-crop rejection, independent field evaluation, and the fixed 60% disease decision rule before deployment.
